# Experiment Analysis
This notebook demonstrates how to load, filter, and analyze the experiment logs using Pandas.
The `ExperimentTracker` seamlessly intercepts runs from both `ExperimentRunner` and `CMAESGameOptimizer` and logs them into specialized files by run type and algorithm (e.g. `outputs/experiments_cmaes_omwu.jsonl`).

In [1]:
import pandas as pd
import json
import matplotlib.pyplot as plt

# Optional: set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

## 1. Load the Data
Because the file is in JSON Lines (JSONL) format, we can load it directly into pandas using `lines=True`. 
We also extract the metrics and config dictionaries into their own columns.

In [2]:
import sys
sys.path.append("..")

from src.utils.tracking import list_available_tables, load_jsonl_table

# 1. Fetch available tables so you know what you can load!
tables = list_available_tables()
print("Available Tables:")
for t in tables:
    print(f" - {t}")

# 2. Easily load specific tables (or use '*' to load all algorithms for a run type)
cmaes_runs = load_jsonl_table("experiments_cmaes_*.jsonl")
print(f"\nLoaded {len(cmaes_runs)} CMA-ES runs.")

dynamic_runs = load_jsonl_table("experiments_dynamic_*.jsonl")
print(f"Loaded {len(dynamic_runs)} Dynamic runs.")


Available Tables:
 - experiments_cmaes-tune_omwu.jsonl
 - experiments_cmaes_omwu.jsonl
 - experiments_dynamic_dmwu.jsonl
 - experiments_dynamic_mwu.jsonl
 - experiments_dynamic_omwu.jsonl

Loaded 275 CMA-ES runs.
Loaded 38587 Dynamic runs.


In [3]:
cmaes_runs

,timestamp,session_id,parent_session_id,run_type,algorithm,eta,total_steps,game_generator,cmaes_objective,cmaes_sigma,config,regret,penalty,fitness,delta,log_reg,peak_weight
0,2026-08-22T09:08:44.663373+00:00,01e94390-b1d2-48f0-a6d0-897731437300,None,cmaes,omwu,0.05,5000,random,envelope_trend_log,0.459012,"{'name': 'random_omwu', 'session_id': '01e9439...",13.604065,0.000000e+00,6.065265e+00,-0.038696,1.689126,NaN
1,2026-08-22T09:08:45.196665+00:00,7d9bf2e8-b3b0-4a40-b3ef-7749d05a7d88,None,cmaes,omwu,0.05,5000,random,envelope_trend_log,0.205580,"{'name': 'random_omwu', 'session_id': '7d9bf2e...",8.597839,0.000000e+00,9.378294e+00,6.674896,1.827862,NaN
2,2026-08-22T09:08:46.101552+00:00,067374f4-7de5-49d8-ab89-4ed62863ef5e,None,cmaes,omwu,0.05,5000,random,envelope_trend_log,0.294300,"{'name': 'random_omwu', 'session_id': '067374f...",13.739563,0.000000e+00,7.260906e+00,0.000061,1.952633,NaN
3,2026-08-22T09:08:46.628132+00:00,8374c2fc-69b3-43c3-bf64-8166807c3a8e,None,cmaes,omwu,0.05,5000,random,envelope_trend_log,0.443027,"{'name': 'random_omwu', 'session_id': '8374c2f...",12.964722,0.000000e+00,7.379511e+00,4.823303,1.722569,NaN
4,2026-08-22T09:08:47.098743+00:00,94db59bd-96f4-477c-8356-66e0b59ebb1b,None,cmaes,omwu,0.05,5000,random,envelope_trend_log,0.417677,"{'name': 'random_omwu', 'session_id': '94db59b...",6.534363,0.000000e+00,5.174530e+00,3.214630,1.153734,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
270,2026-08-30T01:27:20.003433+00:00,ec9bcf6d,None,cmaes,omwu,NaN,400000,random,envelope_trend_log,0.300000,"{'name': 'random_omwu', 'session_id': 'ec9bcf6...",66.416843,1.033258e+07,-1.033246e+07,-17.676559,33.567605,NaN
271,2026-08-30T05:42:05.236713+00:00,2dda8be5,None,cmaes,omwu,NaN,450000,random,delta_reg,0.300000,"{'name': 'random_omwu', 'session_id': '2dda8be...",101.722636,1.072857e+08,-1.072855e+08,-5.730767,41.600249,NaN
272,2026-08-30T07:47:09.758942+00:00,ab109edc,None,cmaes,omwu,NaN,100000,random,delta_reg,0.300000,"{'name': 'random_omwu', 'session_id': 'ab109ed...",6.385824,1.012692e+00,1.025973e+01,2.730328,3.708161,NaN
273,2026-08-30T07:54:54.707576+00:00,2d4613c4,None,cmaes,omwu,NaN,100000,random,envelope_trend,0.300000,"{'name': 'random_omwu', 'session_id': '2d4613c...",10.310818,0.000000e+00,2.565729e+01,-0.178677,NaN,20.621637


## 2. Analyze CMA-ES Training Runs
Let's filter for the CMA-ES runs and see what hyper-parameters led to the best `regret`.

In [4]:
if not cmaes_runs.empty:
    # Sort by the final metric (e.g., regret)
    best_cmaes = cmaes_runs.sort_values(by='regret', ascending=False)
    display(best_cmaes[['session_id', 'algorithm', 'cmaes_sigma', 'eta', 'total_steps', 'regret', 'fitness']].head())


,session_id,algorithm,cmaes_sigma,eta,total_steps,regret,fitness
271,2dda8be5,omwu,0.3,NaN,450000,101.722636,-1.072855e+08
270,ec9bcf6d,omwu,0.3,NaN,400000,66.416843,-1.033246e+07
269,a0cb5a36,omwu,0.3,NaN,400000,66.416843,-1.033196e+07
263,c4baba40,omwu,0.3,NaN,300000,60.628809,4.471502e+02
264,a509cb98,omwu,0.3,NaN,300000,44.204152,7.802168e+01


## 3. Link Validation Runs to CMA-ES
By setting `config.parent_session_id = cmaes_session_id` in your validation scripts, you can easily map the long-horizon dynamic evaluations back to the specific CMA-ES run that generated the matrix.

In [5]:
# We can JOIN (merge) the dynamic runs with their parent CMA-ES runs
if not dynamic_runs.empty and not cmaes_runs.empty:
    # Drop rows without a parent_session_id to avoid merge type errors
    dyn_to_merge = dynamic_runs.dropna(subset=['parent_session_id']).copy()
    
    if not dyn_to_merge.empty:
        # Ensure string type for merge
        dyn_to_merge['parent_session_id'] = dyn_to_merge['parent_session_id'].astype(str)
        cmaes_to_merge = cmaes_runs.copy()
        cmaes_to_merge['session_id'] = cmaes_to_merge['session_id'].astype(str)

        merged = pd.merge(
            dyn_to_merge, 
            cmaes_to_merge, 
            left_on='parent_session_id', 
            right_on='session_id', 
            suffixes=('_val', '_train')
        )
        
        # Now we can see the validation regret next to the training regret
        display_cols = ['session_id_train', 'session_id_val', 'regret', 'final_max_avg_regret']
        
        if all(col in merged.columns for col in display_cols):
            print("Joined Training vs Validation Regrets:")
            display(merged)
        else:
            print("Merge successful, but display columns missing.")
    else:
        print("No dynamic runs with a valid parent_session_id available to merge.")
else:
    print("No runs available to merge yet. Run short_horizon_optimization.ipynb first!")


Joined Training vs Validation Regrets:


,timestamp_val,session_id_val,parent_session_id_val,run_type_val,algorithm_val,eta_val,total_steps_val,game_generator_val,cmaes_objective_val,cmaes_sigma_val,config_val,final_max_avg_regret,timestamp_train,session_id_train,parent_session_id_train,run_type_train,algorithm_train,eta_train,total_steps_train,game_generator_train,cmaes_objective_train,cmaes_sigma_train,config_train,regret,penalty,fitness,delta,log_reg,peak_weight
0,2026-08-22T09:08:43.961888+00:00,3d118115,7d9bf2e8-b3b0-4a40-b3ef-7749d05a7d88,dynamic,omwu,0.05,5000,custom,envelope_trend_log,0.205580,"{'name': 'random_omwu', 'session_id': '3d11811...",0.002794,2026-08-22T09:08:45.196665+00:00,7d9bf2e8-b3b0-4a40-b3ef-7749d05a7d88,None,cmaes,omwu,0.05,5000,random,envelope_trend_log,0.205580,"{'name': 'random_omwu', 'session_id': '7d9bf2e...",8.597839,0.0,9.378294,6.674896,1.827862,NaN
1,2026-08-22T09:08:44.659372+00:00,fa7bc706,01e94390-b1d2-48f0-a6d0-897731437300,dynamic,omwu,0.05,5000,custom,envelope_trend_log,0.459012,"{'name': 'random_omwu', 'session_id': 'fa7bc70...",0.002812,2026-08-22T09:08:44.663373+00:00,01e94390-b1d2-48f0-a6d0-897731437300,None,cmaes,omwu,0.05,5000,random,envelope_trend_log,0.459012,"{'name': 'random_omwu', 'session_id': '01e9439...",13.604065,0.0,6.065265,-0.038696,1.689126,NaN
2,2026-08-22T09:08:44.662372+00:00,3d118115,7d9bf2e8-b3b0-4a40-b3ef-7749d05a7d88,dynamic,omwu,0.05,5000,custom,envelope_trend_log,0.205580,"{'name': 'random_omwu', 'session_id': '3d11811...",0.002795,2026-08-22T09:08:45.196665+00:00,7d9bf2e8-b3b0-4a40-b3ef-7749d05a7d88,None,cmaes,omwu,0.05,5000,random,envelope_trend_log,0.205580,"{'name': 'random_omwu', 'session_id': '7d9bf2e...",8.597839,0.0,9.378294,6.674896,1.827862,NaN
3,2026-08-22T09:08:44.863919+00:00,3d118115,7d9bf2e8-b3b0-4a40-b3ef-7749d05a7d88,dynamic,omwu,0.05,5000,custom,envelope_trend_log,0.205580,"{'name': 'random_omwu', 'session_id': '3d11811...",0.002785,2026-08-22T09:08:45.196665+00:00,7d9bf2e8-b3b0-4a40-b3ef-7749d05a7d88,None,cmaes,omwu,0.05,5000,random,envelope_trend_log,0.205580,"{'name': 'random_omwu', 'session_id': '7d9bf2e...",8.597839,0.0,9.378294,6.674896,1.827862,NaN
4,2026-08-22T09:08:45.176561+00:00,c960e840,067374f4-7de5-49d8-ab89-4ed62863ef5e,dynamic,omwu,0.05,5000,custom,envelope_trend_log,0.294300,"{'name': 'random_omwu', 'session_id': 'c960e84...",0.002800,2026-08-22T09:08:46.101552+00:00,067374f4-7de5-49d8-ab89-4ed62863ef5e,None,cmaes,omwu,0.05,5000,random,envelope_trend_log,0.294300,"{'name': 'random_omwu', 'session_id': '067374f...",13.739563,0.0,7.260906,0.000061,1.952633,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37658,2026-08-30T08:04:01.706836+00:00,8d4ca37d,b0b556fb,dynamic,omwu,NaN,100000,custom,envelope_trend_log,0.300000,"{'name': 'random_omwu', 'session_id': '8d4ca37...",0.000115,2026-08-30T08:04:20.680010+00:00,b0b556fb,None,cmaes,omwu,NaN,100000,random,envelope_trend_log,0.300000,"{'name': 'random_omwu', 'session_id': 'b0b556f...",9.383982,0.0,9.327987,-0.418449,4.478008,NaN
37659,2026-08-30T08:04:06.461279+00:00,8d4ca37d,b0b556fb,dynamic,omwu,NaN,100000,custom,envelope_trend_log,0.300000,"{'name': 'random_omwu', 'session_id': '8d4ca37...",0.000115,2026-08-30T08:04:20.680010+00:00,b0b556fb,None,cmaes,omwu,NaN,100000,random,envelope_trend_log,0.300000,"{'name': 'random_omwu', 'session_id': 'b0b556f...",9.383982,0.0,9.327987,-0.418449,4.478008,NaN
37660,2026-08-30T08:04:11.216847+00:00,8d4ca37d,b0b556fb,dynamic,omwu,NaN,100000,custom,envelope_trend_log,0.300000,"{'name': 'random_omwu', 'session_id': '8d4ca37...",0.000115,2026-08-30T08:04:20.680010+00:00,b0b556fb,None,cmaes,omwu,NaN,100000,random,envelope_trend_log,0.300000,"{'name': 'random_omwu', 'session_id': 'b0b556f...",9.383982,0.0,9.327987,-0.418449,4.478008,NaN
37661,2026-08-30T08:04:15.933800+00:00,8d4ca37d,b0b556fb,dynamic,omwu,NaN,100000,custom,envelope_trend_log,0.300000,"{'name': 'random_omwu', 'session_id': '8d4ca37...",0.000115,20

In [6]:
merged.query('parent_session_id_val == "ec9bcf6d"')

,timestamp_val,session_id_val,parent_session_id_val,run_type_val,algorithm_val,eta_val,total_steps_val,game_generator_val,cmaes_objective_val,cmaes_sigma_val,config_val,final_max_avg_regret,timestamp_train,session_id_train,parent_session_id_train,run_type_train,algorithm_train,eta_train,total_steps_train,game_generator_train,cmaes_objective_train,cmaes_sigma_train,config_train,regret,penalty,fitness,delta,log_reg,peak_weight
37036,2026-08-29T21:37:32.446549+00:00,ca636d90,ec9bcf6d,dynamic,omwu,NaN,400000,custom,envelope_trend_log,0.3,"{'name': 'random_omwu', 'session_id': 'ca636d9...",0.000333,2026-08-30T01:27:20.003433+00:00,ec9bcf6d,None,cmaes,omwu,NaN,400000,random,envelope_trend_log,0.3,"{'name': 'random_omwu', 'session_id': 'ec9bcf6...",66.416843,1.033258e+07,-1.033246e+07,-17.676559,33.567605,NaN
37037,2026-08-29T21:39:59.160942+00:00,ca636d90,ec9bcf6d,dynamic,omwu,NaN,400000,custom,envelope_trend_log,0.3,"{'name': 'random_omwu', 'session_id': 'ca636d9...",0.000333,2026-08-30T01:27:20.003433+00:00,ec9bcf6d,None,cmaes,omwu,NaN,400000,random,envelope_trend_log,0.3,"{'name': 'random_omwu', 'session_id': 'ec9bcf6...",66.416843,1.033258e+07,-1.033246e+07,-17.676559,33.567605,NaN
37038,2026-08-29T21:42:21.684540+00:00,ca636d90,ec9bcf6d,dynamic,omwu,NaN,400000,custom,envelope_trend_log,0.3,"{'name': 'random_omwu', 'session_id': 'ca636d9...",0.000333,2026-08-30T01:27:20.003433+00:00,ec9bcf6d,None,cmaes,omwu,NaN,400000,random,envelope_trend_log,0.3,"{'name': 'random_omwu', 'session_id': 'ec9bcf6...",66.416843,1.033258e+07,-1.033246e+07,-17.676559,33.567605,NaN
37039,2026-08-29T21:44:47.251739+00:00,ca636d90,ec9bcf6d,dynamic,omwu,NaN,400000,custom,envelope_trend_log,0.3,"{'name': 'random_omwu', 'session_id': 'ca636d9...",0.000333,2026-08-30T01:27:20.003433+00:00,ec9bcf6d,None,cmaes,omwu,NaN,400000,random,envelope_trend_log,0.3,"{'name': 'random_omwu', 'session_id': 'ec9bcf6...",66.416843,1.033258e+07,-1.033246e+07,-17.676559,33.567605,NaN
37040,2026-08-29T21:47:08.935354+00:00,ca636d90,ec9bcf6d,dynamic,omwu,NaN,400000,custom,envelope_trend_log,0.3,"{'name': 'random_omwu', 'session_id': 'ca636d9...",0.000333,2026-08-30T01:27:20.003433+00:00,ec9bcf6d,None,cmaes,omwu,NaN,400000,random,envelope_trend_log,0.3,"{'name': 'random_omwu', 'session_id': 'ec9bcf6...",66.416843,1.033258e+07,-1.033246e+07,-17.676559,33.567605,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37131,2026-08-30T01:18:03.057630+00:00,ca636d90,ec9bcf6d,dynamic,omwu,NaN,400000,custom,envelope_trend_log,0.3,"{'name': 'random_omwu', 'session_id': 'ca636d9...",0.000286,2026-08-30T01:27:20.003433+00:00,ec9bcf6d,None,cmaes,omwu,NaN,400000,random,envelope_trend_log,0.3,"{'name': 'random_omwu', 'session_id': 'ec9bcf6...",66.416843,1.033258e+07,-1.033246e+07,-17.676559,33.567605,NaN
37132,2026-08-30T01:20:24.805695+00:00,ca636d90,ec9bcf6d,dynamic,omwu,NaN,400000,custom,envelope_trend_log,0.3,"{'name': 'random_omwu', 'session_id': 'ca636d9...",0.000296,2026-08-30T01:27:20.003433+00:00,ec9bcf6d,None,cmaes,omwu,NaN,400000,random,envelope_trend_log,0.3,"{'name': 'random_omwu', 'session_id': 'ec9bcf6...",66.416843,1.033258e+07,-1.033246e+07,-17.676559,33.567605,NaN
37133,2026-08-30T01:22:46.051735+00:00,ca636d90,ec9bcf6d,dynamic,omwu,NaN,400000,custom,envelope_trend_log,0.3,"{'name': 'random_omwu', 'session_id': 'ca636d9...",0.000292,2026-08-30T01:27:20.003433+00:00,ec9bcf6d,None,cmaes,omwu,NaN,400000,random,envelope_trend_log,0.3,"{'name': 'random_omwu', 'session_id': 'ec9bcf6...",66.416843,1.033258e+07,-1.033246e+07,-17.676559,33.567605,NaN
37134,2026-08-30T01:25:02.275253+00:00,ca636d90,ec9bcf6d,dynamic,omwu,NaN,400000,custom,envelope_trend_log,0.3,"{'name': 'random_omwu', 'session_id': 'ca636d9...",0.000301,2026-08-30T01:27:20.003433+00:00,ec9bcf6d,None,cmaes,omwu,NaN,400000,random,envelope_trend_log,0.3,"{'name': 'random_omwu', 'session_id': 'ec9bcf6...",66.416843,1.033258e+07,-1.033246e+07,-17.676559,